In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
project_root = Path.cwd()
if not (project_root / "db.sqlite3").exists():
    project_root = project_root.parent

database_path = project_root / "db.sqlite3"
csv_path = project_root / "restaurant_data.csv"

query = """
SELECT
    order_item.id AS "Order Item ID",
    restaurant_order.id AS "Order ID",
    restaurant_order.datetime AS "Order Datetime",
    product.id AS "Product ID",
    product.name AS "Product Name",
    product.price AS "Product Price",
    order_item.quantity AS "Quantity"
FROM restaurant_orderitem AS order_item
JOIN restaurant_order
    ON restaurant_order.id = order_item.order_id
JOIN restaurant_product AS product
    ON product.id = order_item.product_id
"""

with sqlite3.connect(database_path) as connection:
    joined_data = pd.read_sql_query(query, connection)

joined_data.to_csv(csv_path, index=False)
df = pd.read_csv(csv_path, parse_dates=["Order Datetime"])
assert len(df) == 74818
df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_products_by_quantity = (
    df.groupby("Product Name")["Quantity"]
    .sum()
    .nlargest(10)
)

top_products_by_quantity.plot.pie(
    autopct="%1.1f%%",
    figsize=(9, 9),
    title="Top 10 Products by Quantity Sold",
)
plt.ylabel("")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["Item Price"] = df["Product Price"] * df["Quantity"]
top_products_by_revenue = (
    df.groupby("Product Name")["Item Price"]
    .sum()
    .nlargest(10)
)

top_products_by_revenue.plot.pie(
    autopct="%1.1f%%",
    figsize=(9, 9),
    title="Top 10 Products by Revenue",
)
plt.ylabel("")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["Order Hour"] = df["Order Datetime"].dt.hour
income_by_hour = (
    df.groupby("Order Hour")["Item Price"]
    .sum()
    .reindex(range(24), fill_value=0)
)

income_by_hour.plot.bar(
    figsize=(12, 6),
    title="Restaurant Income by Order Hour",
    xlabel="Order Hour",
    ylabel="Income",
)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
days_of_week = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
df["Order Day Of The Week"] = df["Order Datetime"].dt.day_name()
income_by_day = (
    df.groupby("Order Day Of The Week")["Item Price"]
    .sum()
    .reindex(days_of_week, fill_value=0)
)

income_by_day.plot.bar(
    figsize=(10, 6),
    title="Restaurant Income by Day of the Week",
    xlabel="Day of the Week",
    ylabel="Income",
)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()